# Programación declarativa @ URJC
# Programación funcional
## Examen de prueba 2 (estilo convocatoria ordinaria, con árboles)

La duración del examen es de 1h:30m.

> Las celdas marcadas con `// IMPLEMENTA TU RESPUESTA` son las que tienes que completar. Ejecuta los tests de cada apartado con la celda `run(...)` que le sigue.

# Preámbulo

In [ ]:
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

### Algunas definiciones de tipos y funciones auxiliares

In [ ]:
object Signatures:
    abstract class List[A]:

        // Common HOFs
        def foldRight[B](nil: B)(cons: (A, B) => B): B
        def foldLeft[B](initial: B)(update: (B, A) => B): B
        def map[B](f: A => B): List[B]
        def flatMap[B](f: A => List[B]): List[B]
        def filter(f: A => Boolean): List[A]
        def forall(pred: A => Boolean): Boolean
        def exists(pred: A => Boolean): Boolean

        // Common functions
        def length: Int
        def reverse: List[A]
        def drop(i: Int): List[A]
        def zip[B](other: List[B]): List[(A, B)]
        def appended(e: A): List[A]

    // List.fill(n)(e) crea una lista con n copias de e:  List.fill(3)('a') == List('a','a','a')

In [ ]:
// type Tree[A] = 1 + Tree[A] * A * Tree[A]

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:

    def void[A]: Tree[A] =
        Empty()

    def leaf[A](a: A): Node[A] =
        Node(Empty(), a, Empty())

    def right[A](a: A, tree: Tree[A]): Node[A] =
        Node(Empty(), a, tree)

    def left[A](tree: Tree[A], a: A): Node[A] =
        Node(tree, a, Empty())

    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] =
        Node(left, a, right)

import Tree._

### Definiciones auxiliares sobre la correspondencia Curry-Howard y Tipos Algebraicos de Datos

In [ ]:
trait Isomorphic[A, B]:

    def from(a: A): B

    def to(b: B): A

    // equality

    def equalA(a1: A, a2: A): Boolean =
        a1 == a2

    def equalB(b1: B, b2: B): Boolean =
        b1 == b2

    // Bijection laws

    def law1(a: A): Boolean =
        equalA(to(from(a)), a)

    def law2(b: B): Boolean =
        equalB(from(to(b)), b)

In [ ]:
type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

# Ejercicio 1 (2 puntos)

#### a) (1 punto)

Utiliza la correspondencia de Curry-Howard para demostrar las siguientes tautologías de la lógica proposicional intuicionista:

$ \vdash (p \rightarrow q \rightarrow r) \rightarrow p \wedge q \rightarrow r$

In [ ]:
// IMPLEMENTA TU RESPUESTA

def proof1[P, Q, R]: (P => Q => R) => ((P, Q)) => R =
    ???

$ \vdash (p \rightarrow q) \rightarrow (p \rightarrow \neg q) \rightarrow \neg p$

In [ ]:
// IMPLEMENTA TU RESPUESTA

def proof2[P, Q]: (P => Q) => (P => Not[Q]) => Not[P] =
    ???

#### b) (1 punto)

Se desea utilizar la correspondencia de Curry-Howard para demostrar la validez de la siguiente tautología de la lógica clásica proposicional:

$ \vdash (\neg q \rightarrow \neg p) \rightarrow p \rightarrow q $

Para ello, utiliza como premisa adicional la ley de la doble negación aplicada a la proposición $q$.

In [ ]:
// IMPLEMENTA TU RESPUESTA

def proof[P, Q](dn: DoubleNegation): (Not[Q] => Not[P]) => P => Q =
    ???

# Ejercicio 2 (2 puntos)

Demuestra que, para cualesquiera tipos `A` y `B`, los tipos de datos `Boolean => (A, B)` y `((A, A), (B, B))` son isomorfos, de las dos formas siguientes:

#### a) (0,5 puntos)

Utilizando la equivalencia entre tipos algebraicos de datos y las operaciones aritméticas.

_Escribe aquí la cadena de igualdades:_

|Boolean => (A, B)| =

...

|((A, A), (B, B))|

#### b) (1,5 puntos)

Implementando una biyección entre ambos tipos de datos mediante una instanciación del trait `Isomorphic`.

In [ ]:
// IMPLEMENTA TU RESPUESTA

class Iso[A, B] extends Isomorphic[Boolean => (A, B), ((A, A), (B, B))]:

    def from(f: Boolean => (A, B)): ((A, A), (B, B)) =
        ???

    def to(t: ((A, A), (B, B))): Boolean => (A, B) =
        ???

Para comprobar la ley 1 hay que comparar funciones, así que el test compara sus resultados en `false` y en `true`:

In [ ]:
class TestIso(iso: Isomorphic[Boolean => (Int, Char), ((Int, Int), (Char, Char))])
extends AnyFlatSpec with should.Matchers:

    "law2" should "hold" in:
        iso.law2(((1, 2), ('a', 'b'))) shouldBe true
        iso.law2(((0, 0), ('z', 'z'))) shouldBe true

    "law1" should "hold" in:
        val f: Boolean => (Int, Char) = b => if b then (1, 'a') else (2, 'b')
        val g = iso.to(iso.from(f))
        g(true) shouldBe f(true)
        g(false) shouldBe f(false)

In [ ]:
run(TestIso(Iso[Int, Char]()))

# Ejercicio 3 (3 puntos)

La función `pairwise` recibe una lista y devuelve la lista de pares formados por cada elemento y el que le sigue. Si la lista tiene menos de dos elementos, devuelve la lista vacía. Por ejemplo:

In [ ]:
class TestPairwise(pairwise: List[Int] => List[(Int, Int)])
extends AnyFlatSpec with should.Matchers:

    "pairwise" should "work" in:
        pairwise(List()) shouldBe List()
        pairwise(List(1)) shouldBe List()
        pairwise(List(1, 2)) shouldBe List((1, 2))
        pairwise(List(1, 2, 3, 4)) shouldBe List((1, 2), (2, 3), (3, 4))
        pairwise(List(5, 3, 5)) shouldBe List((5, 3), (3, 5))

#### a) (0,5 puntos) Implementa la función `pairwise` mediante recursión (no final).

In [ ]:
// IMPLEMENTA TU RESPUESTA

def pairwise[A](l: List[A]): List[(A, A)] =
    ???

In [ ]:
run(TestPairwise(pairwise))

#### b) (1 punto) Implementa la función `pairwise` mediante recursión final.

In [ ]:
// IMPLEMENTA TU RESPUESTA

def pairwise[A](l: List[A]): List[(A, A)] =
    ???

In [ ]:
run(TestPairwise(pairwise))

#### c) (1 punto) Implementa la función `pairwise` mediante la función `foldLeft`. _Pista_: además de la solución parcial, el acumulador deberá guardar el elemento anterior (si existe).

In [ ]:
// IMPLEMENTA TU RESPUESTA

def pairwise[A](l: List[A]): List[(A, A)] =
    ???

In [ ]:
run(TestPairwise(pairwise))

#### d) (0,5 puntos) Implementa la función `pairwise` sin recursión, utilizando exclusivamente las funciones `zip` y `drop`.

In [ ]:
// IMPLEMENTA TU RESPUESTA

def pairwise[A](l: List[A]): List[(A, A)] =
    ???

In [ ]:
run(TestPairwise(pairwise))

# Ejercicio 4 (3 puntos)

Considérense las siguientes funciones sobre árboles binarios, basadas en un mismo patrón de diseño:

In [ ]:
def size[A](t: Tree[A]): Int =
    t match
        case Empty() => 0
        case Node(l, _, r) => size(l) + 1 + size(r)

In [ ]:
def sumTree(t: Tree[Int]): Int =
    t match
        case Empty() => 0
        case Node(l, x, r) => sumTree(l) + x + sumTree(r)

In [ ]:
def depth[A](t: Tree[A]): Int =
    t match
        case Empty() => 0
        case Node(l, _, r) => 1 + math.max(depth(l), depth(r))

In [ ]:
val t1: Tree[Int] = node(leaf(1), 2, right(3, leaf(4)))
val t2: Tree[Int] = left(left(leaf(7), 5), 3)

class TestTree(size: Tree[Int] => Int, sumTree: Tree[Int] => Int, depth: Tree[Int] => Int)
extends AnyFlatSpec with should.Matchers:

    "size" should "work" in:
        size(void) shouldBe 0
        size(t1) shouldBe 4
        size(t2) shouldBe 3

    "sumTree" should "work" in:
        sumTree(void) shouldBe 0
        sumTree(t1) shouldBe 10
        sumTree(t2) shouldBe 15

    "depth" should "work" in:
        depth(void) shouldBe 0
        depth(t1) shouldBe 3
        depth(t2) shouldBe 3

In [ ]:
run(TestTree(size, sumTree, depth))

#### a) (1,5 puntos) Implementa una función de orden superior, denominada `foldTree`, que encapsule el patrón de diseño utilizado en las funciones `size`, `sumTree` y `depth`.

In [ ]:
// IMPLEMENTA TU RESPUESTA

def foldTree[A, B](t: Tree[A])(empty: B)(node: (B, A, B) => B): B =
    ???

#### b) (1 punto) Implementa de una manera más modular las funciones `size`, `sumTree` y `depth` mediante la función `foldTree`.

In [ ]:
// IMPLEMENTA TU RESPUESTA

def size[A](t: Tree[A]): Int =
    ???

def sumTree(t: Tree[Int]): Int =
    ???

def depth[A](t: Tree[A]): Int =
    ???

In [ ]:
run(TestTree(size, sumTree, depth))

#### c) (0,5 puntos) Implementa, mediante `foldTree`, la función `inorder`, que devuelve la lista de los elementos del árbol recorrido en orden (subárbol izquierdo, raíz, subárbol derecho).

In [ ]:
class TestInorder(inorder: Tree[Int] => List[Int]) extends AnyFlatSpec with should.Matchers:

    "inorder" should "work" in:
        inorder(void) shouldBe List()
        inorder(t1) shouldBe List(1, 2, 3, 4)
        inorder(t2) shouldBe List(7, 5, 3)

In [ ]:
// IMPLEMENTA TU RESPUESTA

def inorder[A](t: Tree[A]): List[A] =
    ???

In [ ]:
run(TestInorder(inorder))